# RAG From Scratch (No Vector DB) — Retrieval, Augmentation, Generation

In [1]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import numpy as np

load_dotenv()

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-5.6-luna")

c:\Users\hp\Desktop\genai python\genai_class\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 1. Dummy 20-page book

In [2]:
book_pages = [
    "Page 1: The Solar System has eight planets orbiting the Sun. Mercury is closest to the Sun, while Neptune is farthest away.",
    "Page 2: Photosynthesis is the process plants use to convert sunlight into energy. Chlorophyll in leaves absorbs light and combines it with water and carbon dioxide to produce glucose and oxygen.",
    "Page 3: The French Revolution began in 1789 and ended the monarchy in France. It was driven by financial crisis, social inequality, and Enlightenment ideas about liberty and citizenship.",
    "Page 4: The human digestive system breaks down food into nutrients the body can absorb. It starts in the mouth and ends in the intestines, with the stomach and liver playing key roles.",
    "Page 5: World War II lasted from 1939 to 1945 and involved most of the world's nations. It ended with the defeat of Germany and Japan by the Allied powers.",
    "Page 6: Machine learning is a branch of artificial intelligence where models learn patterns from data. Common approaches include supervised learning, unsupervised learning, and reinforcement learning.",
    "Page 7: Ocean ecosystems support a huge diversity of life, from microscopic plankton to giant whales. Coral reefs, in particular, are some of the most biodiverse habitats on Earth.",
    "Page 8: Ancient Egypt was a civilization along the Nile River known for its pyramids and pharaohs. Hieroglyphics were used as their writing system for thousands of years.",
    "Page 9: Climate change refers to long-term shifts in temperatures and weather patterns, largely driven by burning fossil fuels. It leads to rising sea levels and more extreme weather events.",
    "Page 10: The water cycle describes how water moves through evaporation, condensation, and precipitation. Rivers and oceans continuously exchange water with the atmosphere.",
    "Page 11: The Indian Independence Movement sought to end British colonial rule in India. Leaders like Mahatma Gandhi promoted nonviolent civil disobedience, leading to independence in 1947.",
    "Page 12: DNA carries the genetic instructions for all living organisms. Genes made of DNA determine traits that are passed from parents to offspring.",
    "Page 13: The Renaissance was a period of renewed interest in art, science, and classical learning in Europe. It began in Italy and produced figures like Leonardo da Vinci and Michelangelo.",
    "Page 14: Volcanoes and earthquakes are caused by the movement of tectonic plates beneath Earth's surface. Volcanic eruptions can release lava, ash, and gases with significant force.",
    "Page 15: The internet is a global network of interconnected computers that communicate using standardized protocols. It enables services like email, web browsing, and video streaming.",
    "Page 16: The human circulatory system pumps blood throughout the body via the heart, arteries, and veins. It delivers oxygen and nutrients to cells while removing waste products.",
    "Page 17: The Roman Empire was one of the largest empires in ancient history, spanning Europe, North Africa, and the Middle East. It is known for its engineering, law, and military organization.",
    "Page 18: A balanced diet includes proteins, carbohydrates, fats, vitamins, and minerals. Proper nutrition supports growth, energy levels, and long-term health.",
    "Page 19: Space exploration has advanced from early satellite launches to human moon landings and Mars rovers. Organizations like NASA and private companies continue to push the boundaries of space travel.",
    "Page 20: Artificial intelligence ethics examines the responsible development and use of AI systems. Key concerns include bias, privacy, transparency, and the impact of automation on jobs.",
]

print(len(book_pages))

20


In [3]:
book_pages[19]

'Page 20: Artificial intelligence ethics examines the responsible development and use of AI systems. Key concerns include bias, privacy, transparency, and the impact of automation on jobs.'

## 2. Embed each page (v1...v20)

No vector database and no `.npy` files — the vectors just live in a plain Python list in memory for this notebook session.

In [4]:
page_vectors = embeddings.embed_documents(book_pages)

(v1, v2, v3, v4, v5, v6, v7, v8, v9, v10,
 v11, v12, v13, v14, v15, v16, v17, v18, v19, v20) = page_vectors



In [5]:
len(v5)

1536

In [6]:
v6

[0.0116424560546875,
 0.00717926025390625,
 0.0411376953125,
 -0.0260009765625,
 0.029449462890625,
 0.0489501953125,
 0.0184326171875,
 0.0174560546875,
 -0.0242919921875,
 0.04559326171875,
 0.00600433349609375,
 -0.058441162109375,
 -0.0203704833984375,
 0.0039043426513671875,
 0.01184844970703125,
 0.0155029296875,
 0.0254974365234375,
 0.014373779296875,
 0.018310546875,
 0.00908660888671875,
 -0.0147247314453125,
 0.032196044921875,
 0.03375244140625,
 0.0153961181640625,
 0.0175628662109375,
 -0.02154541015625,
 -0.0005011558532714844,
 0.0479736328125,
 -0.0203704833984375,
 -0.004608154296875,
 0.022613525390625,
 -0.01386260986328125,
 -0.0222320556640625,
 0.0325927734375,
 0.026214599609375,
 -0.0020275115966796875,
 0.01611328125,
 0.010101318359375,
 -0.0423583984375,
 0.025146484375,
 -0.0069580078125,
 -0.0091552734375,
 0.0197601318359375,
 0.06488037109375,
 -0.048583984375,
 -0.0377197265625,
 -0.006092071533203125,
 -0.027496337890625,
 -0.005825042724609375,
 0.097

In [7]:
len(page_vectors)

20

## 3. Cosine similarity (from scratch)

In [8]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [10]:
query = "what is internet"
q_v= embeddings.embed_query(query)


In [11]:
scores=[]
for v in page_vectors:
    cos_score=cosine_similarity(q_v, v)
    scores.append(cos_score)
    print(cos_score)

0.07730483499924572
0.03803788022798783
0.05953725639007799
0.07892707495397794
0.04420817129567472
0.07365004480603649
0.057460066314816836
0.04729217612590457
0.048589115191929926
0.08822183185063771
0.05971402837949828
0.039249026184006386
0.13730221585635635
0.020493511827856862
0.5524361997472117
0.09771375997623029
0.08833054979011494
0.07305689241031489
0.10051248689616038
0.11008514741845421


In [12]:
top_indices=np.argsort(scores)[::-1][:3]
top_indices

array([14, 12, 19])

In [ ]:
[(int(i), float(scores[i]), book_pages[i]) for i in top_indices]

## 4. Retrieval — top-3 nearest pages for a query

Given a user query, embed it and score it against every page vector with cosine similarity. This is a brute-force in-memory search — no FAISS/Chroma/Pinecone, just a Python loop and numpy.

In [13]:
def retrieve_top_k(query, k=3):
    query_vector = embeddings.embed_query(query)
    scores = [cosine_similarity(query_vector, v) for v in page_vectors]
    top_indices = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i]), book_pages[i]) for i in top_indices]

In [16]:
query = "Telle me about ancient civillization"
results = retrieve_top_k(query, k=3)

for idx, score, text in results:
    print(f"[{score:.4f}] {text}")

[0.4171] Page 8: Ancient Egypt was a civilization along the Nile River known for its pyramids and pharaohs. Hieroglyphics were used as their writing system for thousands of years.
[0.3854] Page 17: The Roman Empire was one of the largest empires in ancient history, spanning Europe, North Africa, and the Middle East. It is known for its engineering, law, and military organization.
[0.2409] Page 13: The Renaissance was a period of renewed interest in art, science, and classical learning in Europe. It began in Italy and produced figures like Leonardo da Vinci and Michelangelo.


In [24]:
results

[(7,
  0.4171408303241087,
  'Page 8: Ancient Egypt was a civilization along the Nile River known for its pyramids and pharaohs. Hieroglyphics were used as their writing system for thousands of years.'),
 (16,
  0.3853720429055078,
  'Page 17: The Roman Empire was one of the largest empires in ancient history, spanning Europe, North Africa, and the Middle East. It is known for its engineering, law, and military organization.'),
 (12,
  0.24091552464976024,
  'Page 13: The Renaissance was a period of renewed interest in art, science, and classical learning in Europe. It began in Italy and produced figures like Leonardo da Vinci and Michelangelo.')]

## 5. Augmentation — build the context block

Join the retrieved pages into a single context string that gets inserted into the prompt sent to the LLM.

In [18]:
def build_context(retrieved):
    return "\n\n".join(text for _, _, text in retrieved)

context = build_context(results)
print(context)

Page 8: Ancient Egypt was a civilization along the Nile River known for its pyramids and pharaohs. Hieroglyphics were used as their writing system for thousands of years.

Page 17: The Roman Empire was one of the largest empires in ancient history, spanning Europe, North Africa, and the Middle East. It is known for its engineering, law, and military organization.

Page 13: The Renaissance was a period of renewed interest in art, science, and classical learning in Europe. It began in Italy and produced figures like Leonardo da Vinci and Michelangelo.


In [ ]:
context

In [19]:
system_message = SystemMessage(
    content=(
        "Answer using only the provided context. "
        "Do not make up or assume any information. "
        "If the answer is not in the context, say 'I don't know.'"
    )
)

human_message = HumanMessage(
    content=(
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        "Answer:"
    )
)
msg=[system_message, human_message]
msg

[SystemMessage(content="Answer using only the provided context. Do not make up or assume any information. If the answer is not in the context, say 'I don't know.'", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Context:\nPage 8: Ancient Egypt was a civilization along the Nile River known for its pyramids and pharaohs. Hieroglyphics were used as their writing system for thousands of years.\n\nPage 17: The Roman Empire was one of the largest empires in ancient history, spanning Europe, North Africa, and the Middle East. It is known for its engineering, law, and military organization.\n\nPage 13: The Renaissance was a period of renewed interest in art, science, and classical learning in Europe. It began in Italy and produced figures like Leonardo da Vinci and Michelangelo.\n\nQuestion: Telle me about ancient civillization\n\nAnswer:', additional_kwargs={}, response_metadata={})]

## 6. Generation — system message + LLM call

The system message constrains the model to answer only from the retrieved book excerpts, and the human message carries the augmented context plus the original query.

In [20]:
response = llm.invoke(msg)

print(response.content)

Ancient civilizations included:

- **Ancient Egypt**, which developed along the Nile River and was known for pyramids, pharaohs, and hieroglyphic writing.
- **The Roman Empire**, one of the largest ancient empires, spanning Europe, North Africa, and the Middle East. It was known for engineering, law, and military organization.


## 7. End-to-end RAG function

Wraps retrieval, augmentation, and generation into a single call — the full RAG pipeline, no vector DB, no saved embeddings, everything in memory.

In [21]:
def rag_answer(user_query, k=3):
    # Retrieval
    retrieved = retrieve_top_k(user_query, k)

    # Augmentation
    context = build_context(retrieved)

    # Generation
    system_message = SystemMessage(
        content=(
            "Answer using only the provided context. "
            "Do not make up or assume any information. "
            "If the answer is not in the context, say 'I don't know.'"
        )
    )

    human_message = HumanMessage(
        content=(
            f"Context:\n{context}\n\n"
            f"Question: {user_query}\n\n"
            "Answer:"
        )
    )

    response = llm.invoke([system_message, human_message])

    return response.content

In [25]:

q1="Who led India's independence movement?"
q2="What causes earthquakes?"
q3="What are the risks of AI according to the book?"


In [26]:
rag_answer(q1, k=3)

'Mahatma Gandhi led India’s independence movement.'

In [27]:
rag_answer(q2, k=3)

'The movement of tectonic plates beneath Earth’s surface causes earthquakes.'

In [28]:
rag_answer(q3, k=3)

'The risks of AI include bias, privacy concerns, lack of transparency, and the impact of automation on jobs.'